# Hybrid Retrieval (Tri-Hybrid RRF) dan Compact QDA Reranking untuk Segmen Peraturan Presiden

**Kelompok 7 — Temu Kembali Informasi**

| Anggota | NIM |
|---|---|
| Nazwa Abelia Nabira | 09021182328029 |
| Muhammad Safarudin Pratama | 09021282328102 |
| Kenz Raki Abdurrazak | 09021282328117 |
| M. Rasyid Ridho | 09021282328119 |

Implementasi khusus konfigurasi **`2_Stage1+CompactQDA`**:
- **Stage 1:** Weighted Tri-Hybrid RRF (`BM25 k1=0.9, b=0.75 + TF-IDF word & char` + `intfloat/multilingual-e5-large` + `BAAI/bge-m3`, `TOP_K = 30`).
- **Stage 2:** `CompactQDA` (`1024 -> 128`, 4 kepala, `263.809` parameter) di atas token embedding `e5-large` + fusi skor konveks bersih.

## 1. Konfigurasi

In [3]:
%pip -q install rank-bm25 sentence-transformers transformers scikit-learn

import copy, gc, json, random, re, time
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModel, AutoTokenizer
from IPython.display import display

SEED = 42
possible_dirs = [
    Path('/kaggle/input/datasets/gioncarlo11/nlp-ir'),
    Path('/kaggle/input/perpres/dataset'),
    Path('/kaggle/input/perpres'),
    Path('/kaggle/input/datasets/msafarudinp/perpres/dataset'),
    Path('./dataset'),
    Path('../dataset'),
    Path('.')
]
DATA_DIR = next((p for p in possible_dirs if (p / 'dataset_korpus_per_ayat.csv').is_file()), None)
OUT_DIR = Path('/kaggle/working/perpres_compact_qda') if Path('/kaggle/working').is_dir() else Path('./perpres_compact_qda')
OUT_DIR.mkdir(parents=True, exist_ok=True)

E5_MODEL = 'intfloat/multilingual-e5-large'
BGE_MODEL = 'BAAI/bge-m3'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BM25_K1 = 0.9
BM25_B = 0.75
TOP_K = 30
RRF_K = 60
S1_WEIGHTS = {'lex': 1.2, 'e5_large': 1.0, 'bge_m3': 1.0}

MAX_Q_LEN = 64
MAX_D_LEN = 256
BATCH_SIZE_ENCODE = 16 if DEVICE.type == 'cuda' else 4

QDA_DIM = 128
QDA_HEADS = 4
DROPOUT = 0.25
EPOCHS = 5
TRAIN_BATCH_SIZE = 8
LR = 1e-4
WEIGHT_DECAY = 0.01

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE, '| Dataset:', DATA_DIR, '| OUT_DIR:', OUT_DIR)
assert DATA_DIR is not None, 'File dataset_korpus_per_ayat.csv tidak ditemukan di direktori input.'

Note: you may need to restart the kernel to use updated packages.
Device: cuda | Dataset: /kaggle/input/datasets/gioncarlo11/nlp-ir | OUT_DIR: /kaggle/working/perpres_compact_qda


## 2. Pembacaan dan pemeriksaan data

Memuat korpus, query literal, query semantik, dan qrels serta memverifikasi integritas ID dan label.

In [4]:
corpus = pd.read_csv(DATA_DIR / 'dataset_korpus_per_ayat.csv', sep=';')
literal = pd.read_csv(DATA_DIR / 'ds_literal_final.csv', sep=';')
semantic = pd.read_csv(DATA_DIR / 'ds_semantic_final.csv', sep=';')
qrels = pd.read_csv(DATA_DIR / 'qrels_final.csv', sep=';')

queries = pd.concat([
    literal[['row_id', 'query_id', 'query_literal']].rename(columns={'query_literal': 'query_text'}).assign(query_type='Literal'),
    semantic[['row_id', 'query_id', 'query_semantic']].rename(columns={'query_semantic': 'query_text'}).assign(query_type='Semantic'),
], ignore_index=True)

for frame, cols in [(corpus, ['row_id', 'value']), (queries, ['row_id', 'query_id', 'query_text']), (qrels, ['query_id', 'doc_id', 'label'])]:
    assert frame[cols].notna().all().all(), f'Nilai kosong pada {cols}'
assert corpus.row_id.is_unique and queries.query_id.is_unique
assert set(qrels.doc_id).issubset(set(corpus.row_id))
assert set(qrels.query_id).issubset(set(queries.query_id))
assert set(qrels.label.unique()).issubset({0, 1, 2})
assert not qrels.duplicated(['query_id', 'doc_id']).any()
assert len(corpus) == 1509 and len(queries) == 204 and len(qrels) == 3602

display(pd.DataFrame({'komponen': ['Segmen korpus', 'Query literal', 'Query semantik', 'Qrels'],
                      'jumlah': [len(corpus), len(literal), len(semantic), len(qrels)]}))
display(qrels.label.value_counts().sort_index().rename('jumlah').to_frame())

,komponen,jumlah
0,Segmen korpus,1509
1,Query literal,102
2,Query semantik,102
3,Qrels,3602


,jumlah
label,
0,1729
1,1206
2,667


## 3. Pembagian data

Membagi query ke train, validation, dan test berdasarkan ID segmen sumber agar tidak terjadi kebocoran antara query literal dan semantik.

In [5]:
source_ids = sorted(queries.row_id.unique())
rng = np.random.default_rng(SEED)
rng.shuffle(source_ids)
n_train = int(0.70 * len(source_ids))
n_val = int(0.15 * len(source_ids))
source_split = {s: ('train' if i < n_train else 'validation' if i < n_train + n_val else 'test')
                for i, s in enumerate(source_ids)}
queries['split'] = queries.row_id.map(source_split)
qrels = qrels.merge(queries[['query_id', 'split']], on='query_id', validate='many_to_one')
assert queries.groupby('row_id').split.nunique().eq(1).all()

display(pd.crosstab(queries.split, queries.query_type))
queries.to_csv(OUT_DIR / 'query_split.csv', index=False)

query_type,Literal,Semantic
split,,
test,15,15
train,69,75
validation,18,12


## 4. Indeks leksikal dan representasi dense

Membangun indeks leksikal (`BM25 k1=0.9, b=0.75` + `TF-IDF` word & char) serta mengekstrak embedding beku dari `intfloat/multilingual-e5-large` dan `BAAI/bge-m3`.

In [6]:
def tokenize(text):
    return re.findall(r'\w+', str(text).lower(), flags=re.UNICODE)

doc_ids = corpus.row_id.astype(str).tolist()
doc_texts = corpus.value.astype(str).tolist()
query_ids = queries.query_id.astype(str).tolist()
query_texts = queries.query_text.astype(str).tolist()
doc_index = {d: i for i, d in enumerate(doc_ids)}
query_index = {q: i for i, q in enumerate(query_ids)}

# 1. Lexical index (BM25 + TF-IDF word & char)
bm25 = BM25Okapi([tokenize(t) for t in doc_texts], k1=BM25_K1, b=BM25_B)
vec_word = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)
X_word = vec_word.fit_transform(doc_texts)
X_char = vec_char.fit_transform(doc_texts)

def compute_lexical(query_text):
    s_bm = bm25.get_scores(tokenize(query_text))
    s_w = (X_word @ vec_word.transform([query_text]).T).toarray().ravel()
    s_c = (X_char @ vec_char.transform([query_text]).T).toarray().ravel()
    return 0.5 * (s_bm / (s_bm.max() + 1e-8)) + 0.25 * s_w + 0.25 * s_c

lex_cache = {qid: compute_lexical(query_texts[query_index[qid]]) for qid in query_ids}

# 2. Dense encoders (e5-large + bge-m3)
@torch.inference_mode()
def encode_encoder(model_name, texts, max_length, pooling='mean', prefix='', return_tokens=False):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    if DEVICE.type == 'cuda':
        mdl = mdl.half()
    prefixed = [prefix + t for t in texts] if prefix else texts
    pooled, states, masks = [], [], []
    for start in range(0, len(prefixed), BATCH_SIZE_ENCODE):
        batch = tok(prefixed[start:start + BATCH_SIZE_ENCODE], padding='max_length',
                    truncation=True, max_length=max_length, return_tensors='pt')
        mask = batch['attention_mask']
        out = mdl(**{k: v.to(DEVICE) for k, v in batch.items()}).last_hidden_state
        if pooling == 'cls':
            vec = out[:, 0].float()
        else:
            m = mask.to(DEVICE).unsqueeze(-1).float()
            vec = (out.float() * m).sum(1) / m.sum(1).clamp_min(1)
        pooled.append(F.normalize(vec, p=2, dim=1).cpu())
        if return_tokens:
            states.append(out.cpu().to(torch.float16))
            masks.append(mask.bool())
    del mdl, tok
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    if return_tokens:
        return torch.cat(pooled), torch.cat(states), torch.cat(masks)
    return torch.cat(pooled)

e5_doc_dense, doc_states, doc_masks = encode_encoder(E5_MODEL, doc_texts, MAX_D_LEN, pooling='mean', prefix='passage: ', return_tokens=True)
e5_query_dense, query_states, query_masks = encode_encoder(E5_MODEL, query_texts, MAX_Q_LEN, pooling='mean', prefix='query: ', return_tokens=True)

bge_doc_dense = encode_encoder(BGE_MODEL, doc_texts, MAX_D_LEN, pooling='cls', prefix='', return_tokens=False)
bge_query_dense = encode_encoder(BGE_MODEL, query_texts, MAX_Q_LEN, pooling='cls', prefix='', return_tokens=False)

ENC_DIM = int(doc_states.shape[-1])
print('Token states (e5-large):', tuple(doc_states.shape), '| Query:', tuple(query_states.shape))

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (13 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Token states (e5-large): (1509, 256, 1024) | Query: (204, 64, 1024)


## 5. Stage 1: Weighted Tri-Hybrid RRF

Menggabungkan Top-30 hasil dari `lex` (`BM25 + TF-IDF`), `e5-large`, dan `bge-m3` dengan bobot `1.2 : 1.0 : 1.0` menjadi maksimal 20 kandidat unik beserta skor RRF Stage 1.

In [7]:
def retrieve(query_id):
    qi = query_index[query_id]
    score_map = {
        'lex': lex_cache[query_id],
        'e5_large': (e5_doc_dense @ e5_query_dense[qi]).numpy(),
        'bge_m3': (bge_doc_dense @ bge_query_dense[qi]).numpy(),
    }
    fused = defaultdict(float)
    for key, w in S1_WEIGHTS.items():
        ranking = np.argsort(-score_map[key], kind='stable')[:TOP_K]
        for rank, di in enumerate(ranking, start=1):
            fused[int(di)] += w / (RRF_K + rank)
    sorted_cands = sorted(fused.items(), key=lambda x: (-x[1], doc_ids[x[0]]))[:20]
    cand_ids = [doc_ids[di] for di, _ in sorted_cands]
    cand_scores = {doc_ids[di]: score for di, score in sorted_cands}
    return cand_ids, cand_scores

retrieved = {qid: retrieve(qid) for qid in query_ids}
candidates = {qid: v[0] for qid, v in retrieved.items()}
candidate_s1 = {qid: v[1] for qid, v in retrieved.items()}
assert all(1 <= len(v) <= 20 and len(v) == len(set(v)) for v in candidates.values())
print('Rata-rata kandidat:', round(np.mean([len(v) for v in candidates.values()]), 2))

Rata-rata kandidat: 20.0


## 6. Pelatihan `CompactQDA` (Stage 2)

Memproyeksikan representasi token `e5-large` (`1024 -> 128`) lalu melatih modul cross-attention dengan *balanced hard-negative pairs* dan *gain-weighted pairwise loss*.

In [8]:
class CompactQDA(nn.Module):
    def __init__(self, enc_dim=ENC_DIM, dim=QDA_DIM, heads=QDA_HEADS, dropout=DROPOUT):
        super().__init__()
        self.proj = nn.Linear(enc_dim, dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, 2 * dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(2 * dim, dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1)

    def forward(self, q, qm, d, dm):
        qp = F.dropout(self.proj(q), p=DROPOUT, training=self.training)
        dp = F.dropout(self.proj(d), p=DROPOUT, training=self.training)
        attn_out, _ = self.attn(qp, dp, dp, key_padding_mask=~dm, need_weights=False)
        x = self.norm1(qp + attn_out)
        x = self.norm2(x + self.ffn(x))
        pooled = (x * qm.unsqueeze(-1)).sum(1) / qm.sum(1, keepdim=True).clamp_min(1)
        return self.score(pooled).squeeze(-1)

model = CompactQDA().to(DEVICE)
qda_params = sum(p.numel() for p in model.parameters())
print('Parameter trainable CompactQDA:', qda_params)

judgments = {qid: dict(zip(g.doc_id, g.label)) for qid, g in qrels.groupby('query_id')}
train_qids = queries.loc[queries.split.eq('train'), 'query_id'].tolist()
val_qids = queries.loc[queries.split.eq('validation'), 'query_id'].tolist()
test_qids = queries.loc[queries.split.eq('test'), 'query_id'].tolist()

def make_pairs(epoch, max_per_query=24, hard_quota=16):
    rand = random.Random(SEED + epoch)
    pairs = []
    for qid in train_qids:
        cand_set = set(candidates[qid])
        items = list(judgments[qid].items())
        hard_pairs, other_pairs = [], []
        for pos, pos_label in items:
            for neg, neg_label in items:
                if pos_label > neg_label:
                    weight = float((2 ** int(pos_label)) - (2 ** int(neg_label)))
                    item = (query_index[qid], doc_index[pos], doc_index[neg], weight)
                    if neg in cand_set:
                        hard_pairs.append(item)
                    else:
                        other_pairs.append(item)
        rand.shuffle(hard_pairs); rand.shuffle(other_pairs)
        selected = hard_pairs[:hard_quota]
        if len(selected) < max_per_query:
            selected += other_pairs[:max_per_query - len(selected)]
        pairs.extend(selected)
    rand.shuffle(pairs)
    return pairs

def batch_score(qi, di):
    qi = torch.as_tensor(qi, dtype=torch.long)
    di = torch.as_tensor(di, dtype=torch.long)
    q = query_states.index_select(0, qi).to(DEVICE, dtype=torch.float32)
    qm = query_masks.index_select(0, qi).to(DEVICE)
    d = doc_states.index_select(0, di).to(DEVICE, dtype=torch.float32)
    dm = doc_masks.index_select(0, di).to(DEVICE)
    return model(q, qm, d, dm)

@torch.inference_mode()
def score_qda(qid, doc_list, batch_size=32):
    model.eval()
    scores = []
    for start in range(0, len(doc_list), batch_size):
        part = doc_list[start:start + batch_size]
        qi = [query_index[qid]] * len(part)
        di = [doc_index[d] for d in part]
        scores.extend(batch_score(qi, di).cpu().numpy().tolist())
    return np.array(scores, dtype=float)

def fuse_ranking(doc_list, qda_scores, s1_scores, alpha):
    if alpha <= 0.0:
        return [d for d, _ in sorted(zip(doc_list, s1_scores.tolist()), key=lambda z: (-z[1], z[0]))]
    qda_std = qda_scores.std()
    qda_norm = (qda_scores - qda_scores.mean()) / (qda_std + 1e-8) if qda_std > 0 else np.zeros_like(qda_scores)
    s1_std = s1_scores.std()
    s1_norm = (s1_scores - s1_scores.mean()) / (s1_std + 1e-8) if s1_std > 0 else np.zeros_like(s1_scores)
    fused = alpha * qda_norm + (1.0 - alpha) * s1_norm
    return [d for d, _ in sorted(zip(doc_list, fused.tolist()), key=lambda z: (-z[1], z[0]))]

@torch.inference_mode()
def rerank(qid, doc_list, s1_scores=None, batch_size=32, alpha=None):
    if alpha is None:
        alpha = RERANK_ALPHA
    if s1_scores is None:
        s1_scores = candidate_s1[qid]
    s1_arr = np.array([s1_scores[d] for d in doc_list], dtype=float)
    if alpha <= 0.0:
        return fuse_ranking(doc_list, np.zeros_like(s1_arr), s1_arr, 0.0)
    qda_arr = score_qda(qid, doc_list, batch_size=batch_size)
    return fuse_ranking(doc_list, qda_arr, s1_arr, alpha)

RERANK_ALPHA = 0.10
print('Contoh jumlah pasangan latih per epoch:', len(make_pairs(0)))

Parameter trainable CompactQDA: 263809
Contoh jumlah pasangan latih per epoch: 3330


## 7. Validasi model

Memilih checkpoint terbaik dan bobot fusi `alpha` pada validation set menggunakan skor yang di-cache.

In [9]:
def metrics_for(qid, ranking):
    labels = judgments[qid]
    gains = np.array([2**int(labels.get(d, 0)) - 1 for d in ranking[:10]], float)
    dcg = np.sum(gains / np.log2(np.arange(2, len(gains) + 2)))
    ideal = sorted((2**int(v) - 1 for v in labels.values()), reverse=True)[:10]
    idcg = np.sum(np.array(ideal) / np.log2(np.arange(2, len(ideal) + 2)))
    first = next((i for i, d in enumerate(ranking[:10], 1) if labels.get(d, 0) >= 1), None)
    positives = {d for d, v in labels.items() if v >= 1}
    return {
        'nDCG@10': dcg / idcg if idcg else 0.0,
        'MRR@10': 1 / first if first else 0.0,
        'Precision@1': float(bool(ranking) and labels.get(ranking[0], 0) >= 1),
        'Recall@20': len(positives.intersection(ranking[:20])) / len(positives) if positives else 0.0,
    }

val_s1_cache = {qid: np.array([candidate_s1[qid][d] for d in candidates[qid]], dtype=float) for qid in val_qids}

def evaluate_val_cached(alphas):
    qda_cache = {qid: score_qda(qid, candidates[qid]) for qid in val_qids}
    best_a, best_ndcg = None, -1.0
    for a in alphas:
        vals = [metrics_for(qid, fuse_ranking(candidates[qid], qda_cache[qid], val_s1_cache[qid], float(a)))['nDCG@10']
                for qid in val_qids]
        mean_ndcg = float(np.mean(vals))
        if mean_ndcg > best_ndcg:
            best_ndcg, best_a = mean_ndcg, float(a)
    return best_a, best_ndcg, qda_cache

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
best_score = -1.0
history = []
checkpoint = OUT_DIR / 'qda_best.pt'

for epoch in range(EPOCHS):
    model.train(); losses = []
    pairs = make_pairs(epoch)
    for start in range(0, len(pairs), TRAIN_BATCH_SIZE):
        batch = pairs[start:start + TRAIN_BATCH_SIZE]
        qi, pos, neg, w = zip(*batch)
        w_t = torch.as_tensor(w, dtype=torch.float32, device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pair_loss = F.softplus(-(batch_score(qi, pos) - batch_score(qi, neg)))
        loss = (pair_loss * w_t).sum() / w_t.sum().clamp_min(1e-8)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    ep_alpha, ep_ndcg, _ = evaluate_val_cached(np.linspace(0.05, 1.0, 20))
    history.append({'epoch': epoch + 1, 'train_loss': np.mean(losses), 'best_val_alpha': ep_alpha, 'validation_nDCG@10': ep_ndcg})
    print(f'Epoch {epoch+1}: loss={np.mean(losses):.4f}; validation nDCG@10={ep_ndcg:.4f} (alpha={ep_alpha:.2f})')
    if ep_ndcg > best_score:
        best_score = ep_ndcg
        torch.save({'state_dict': model.state_dict(), 'enc_dim': ENC_DIM, 'qda_dim': QDA_DIM,
                    'qda_heads': QDA_HEADS, 'dropout': DROPOUT, 'seed': SEED, 'epoch': epoch + 1}, checkpoint)

display(pd.DataFrame(history))
saved = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
model.load_state_dict(saved['state_dict'])
print('Checkpoint terpilih: epoch', saved['epoch'])

_, _, val_qda_cache = evaluate_val_cached([1.0])
alpha_results = []
for alpha in np.linspace(0.0, 1.0, 21):
    vals = [metrics_for(qid, fuse_ranking(candidates[qid], val_qda_cache[qid], val_s1_cache[qid], float(alpha)))['nDCG@10']
            for qid in val_qids]
    alpha_results.append({'alpha': float(alpha), 'validation_nDCG@10': float(np.mean(vals))})
alpha_table = pd.DataFrame(alpha_results)
best_row = alpha_table.sort_values(['validation_nDCG@10', 'alpha'], ascending=[False, True]).iloc[0]
RERANK_ALPHA = float(best_row.alpha)
display(alpha_table)
print(f'Bobot QDA terpilih: {RERANK_ALPHA:.2f}; validation nDCG@10={best_row["validation_nDCG@10"]:.4f}')

Epoch 1: loss=0.5707; validation nDCG@10=0.7160 (alpha=0.05)
Epoch 2: loss=0.4407; validation nDCG@10=0.7237 (alpha=0.10)
Epoch 3: loss=0.3997; validation nDCG@10=0.7228 (alpha=0.10)
Epoch 4: loss=0.3634; validation nDCG@10=0.7245 (alpha=0.10)
Epoch 5: loss=0.3485; validation nDCG@10=0.7239 (alpha=0.10)


,epoch,train_loss,best_val_alpha,validation_nDCG@10
0,1,0.570736,0.05,0.716042
1,2,0.440686,0.10,0.723731
2,3,0.399678,0.10,0.722811
3,4,0.363433,0.10,0.724514
4,5,0.348463,0.10,0.723890


Checkpoint terpilih: epoch 4


,alpha,validation_nDCG@10
0,0.00,0.712748
1,0.05,0.709111
2,0.10,0.724514
3,0.15,0.719552
4,0.20,0.712198
5,0.25,0.680552
6,0.30,0.665062
7,0.35,0.652577
8,0.40,0.626457
9,0.45,0.600398


Bobot QDA terpilih: 0.10; validation nDCG@10=0.7245


## 8. Evaluasi akhir

Membandingkan `Stage 1 (Tri-Hybrid RRF)` dan `Stage 1 + CompactQDA` pada test set serta menyimpan seluruh artefak model.

In [10]:
@torch.inference_mode()
def stage1_search(query_id):
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    initial, _ = retrieve(query_id)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return initial, (time.perf_counter() - start) * 1000

@torch.inference_mode()
def full_search(query_id):
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    initial, s1_scores = retrieve(query_id)
    final = rerank(query_id, initial, s1_scores=s1_scores)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return final, (time.perf_counter() - start) * 1000

_ = stage1_search(test_qids[0])
_ = full_search(test_qids[0])
records, ranking_rows = [], []
for qid in test_qids:
    initial, s1_latency = stage1_search(qid)
    base = metrics_for(qid, initial)
    final, qda_latency = full_search(qid)
    result = metrics_for(qid, final)
    qtype = queries.iloc[query_index[qid]].query_type
    records += [
        dict(query_id=qid, query_type=qtype, system='1_Stage1_TriHybrid_RRF', **base, latency_ms=s1_latency),
        dict(query_id=qid, query_type=qtype, system='2_Stage1+CompactQDA', **result, latency_ms=qda_latency),
    ]
    ranking_rows += [
        dict(query_id=qid, rank=i + 1, doc_id=d, judged_label=judgments[qid].get(d, np.nan))
        for i, d in enumerate(final)
    ]

results = pd.DataFrame(records)
columns = ['nDCG@10', 'MRR@10', 'Precision@1', 'Recall@20', 'latency_ms']
overall = results.groupby('system')[columns].mean().round(4)
by_type = results.groupby(['system', 'query_type'])[columns].mean().round(4)
display(overall); display(by_type)

results.to_csv(OUT_DIR / 'test_metrics_per_query.csv', index=False)
pd.DataFrame(ranking_rows).to_csv(OUT_DIR / 'test_rankings.csv', index=False)
pd.DataFrame(history).to_csv(OUT_DIR / 'training_history.csv', index=False)
with open(OUT_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'seed': SEED, 'e5_model': E5_MODEL, 'bge_model': BGE_MODEL,
        'bm25_k1': BM25_K1, 'bm25_b': BM25_B, 'top_k_each': TOP_K, 'rrf_k': RRF_K,
        's1_weights': S1_WEIGHTS, 'max_q_len': MAX_Q_LEN, 'max_d_len': MAX_D_LEN,
        'enc_dim': ENC_DIM, 'qda_dim': QDA_DIM, 'qda_heads': QDA_HEADS,
        'dropout': DROPOUT, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
        'train_batch_size': TRAIN_BATCH_SIZE, 'qda_params': qda_params,
        'rerank_alpha': RERANK_ALPHA, 'best_epoch': saved['epoch'],
    }, f, indent=2)
torch.save({
    'doc_states': doc_states, 'doc_masks': doc_masks,
    'e5_doc_dense': e5_doc_dense, 'bge_doc_dense': bge_doc_dense,
    'doc_ids': doc_ids, 'e5_model': E5_MODEL, 'bge_model': BGE_MODEL, 'max_d_len': MAX_D_LEN,
}, OUT_DIR / 'doc_artifacts.pt')
print('Artefak tersimpan:', [p.name for p in sorted(OUT_DIR.iterdir())])

,nDCG@10,MRR@10,Precision@1,Recall@20,latency_ms
system,,,,,
1_Stage1_TriHybrid_RRF,0.6375,0.8317,0.8,0.7315,1.2168
2_Stage1+CompactQDA,0.6387,0.8317,0.8,0.7315,11.9731


nDCG@10  MRR@10  Precision@1  Recall@20  \
system                 query_type                                            
1_Stage1_TriHybrid_RRF Literal      0.8561  1.0000          1.0     0.8871   
                       Semantic     0.4188  0.6633          0.6     0.5759   
2_Stage1+CompactQDA    Literal      0.8555  1.0000          1.0     0.8871   
                       Semantic     0.4219  0.6633          0.6     0.5759   

                                   latency_ms  
system                 query_type              
1_Stage1_TriHybrid_RRF Literal         1.2507  
                       Semantic        1.1829  
2_Stage1+CompactQDA    Literal        12.0739  
                       Semantic       11.8722

Artefak tersimpan: ['config.json', 'doc_artifacts.pt', 'qda_best.pt', 'query_split.csv', 'test_metrics_per_query.csv', 'test_rankings.csv', 'training_history.csv']


## 9. Contoh hasil pencarian

Menampilkan peringkat dokumen untuk satu query uji beserta label relevansi yang tersedia.

In [11]:
def show_results(query_id, top_n=10):
    assert query_id in query_index
    ranking, _ = full_search(query_id)
    rows = []
    for rank, did in enumerate(ranking[:top_n], 1):
        idx = doc_index[did]
        rows.append({
            'rank': rank,
            'doc_id': did,
            'peraturan': getattr(corpus.iloc[idx], 'nomor_peraturan', 'N/A'),
            'segmen': getattr(corpus.iloc[idx], 'sub_segmen', 'N/A'),
            'label': judgments[query_id].get(did, np.nan),
            'teks': doc_texts[idx][:350],
        })
    print('Query:', query_texts[query_index[query_id]])
    display(pd.DataFrame(rows))

show_results(test_qids[0])

Query: pelatihan diselenggarakan secara daring dan luring


,rank,doc_id,peraturan,segmen,label,teks
0,1,ayat_1307,nomor 36 tahun 2020,pasal 5 - ayat (3),2.0,pelatihan sebagaimana dimaksud pada ayat 2 dap...
1,2,ayat_1308,nomor 36 tahun 2020,pasal 6 - ayat (1),0.0,pelatihan sebagaimana dimaksud dalam pasal 5 d...
2,3,ayat_1316,nomor 36 tahun 2020,pasal 10 - ayat (2),1.0,pendaftaran program kartu prakerja dilakukan s...
3,4,ayat_1309,nomor 36 tahun 2020,pasal 6 - ayat (2),NaN,lembaga pelatihan sebagaimana dimaksud pada ay...
4,5,ayat_1319,nomor 36 tahun 2020,pasal 11 - ayat (3),2.0,penerima kartu prakerja sebagaimana dimaksud p...
5,6,ayat_1310,nomor 36 tahun 2020,pasal 7 - teks,0.0,lembaga pelatihan sebagaimana dimaksud dalam p...
6,7,ayat_1306,nomor 36 tahun 2020,pasal 5 - ayat (2),NaN,pelatihan sebagaimana dimaksud pada ayat 1 mel...
7,8,ayat_1305,nomor 36 tahun 2020,pasal 5 - ayat (1),NaN,penerima kartu prakerja berhak mendapatkan ban...
8,9,ayat_844,nomor 197 tahun 2024,pasal 37 - teks,NaN,"pembinaan dan pengelolaan sumber daya manusia,..."
9,10,ayat_1313,nomor 36 tahun 2020,pasal 9 - ayat (1),0.0,ketentuan lebih lanjut mengenai persyaratan le...
